# Baseline model — Demo 1

Evaluation and explanations for the baseline (TF-IDF + logistic regression on HPI text only). The model settings and split are identical to `train_baseline.ipynb`; this notebook does not change or overwrite `models/baseline_model.pkl`.

Every cell prints aggregate numbers, plots, or made-up examples only — no patient rows or note text.

## 1. Setup and data

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Find the repo root so paths work from any working directory.
ROOT = Path.cwd()
while not (ROOT / "app.py").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Run this notebook from inside the CSE4109 repo.")
    ROOT = ROOT.parent

# Data stays outside the repo.
DATA_PATH = Path.home() / "Desktop" / "triage_level.csv"
MODEL_PATH = ROOT / "models" / "baseline_model.pkl"
REPORTS_DIR = ROOT / "reports"

LABELS = [1, 2, 3, 4]
LABEL_NAMES = ["ESI 1", "ESI 2", "ESI 3", "ESI 4"]

print("Imports OK")

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df.shape}")

# Same filtering as train_baseline.ipynb
df = df.dropna(subset=["HPI", "triage"])
print(f"After dropping nulls: {df.shape}")

print("\nTriage distribution:")
print(df["triage"].value_counts().sort_index())

## 2. Same split, same model

The split and pipeline below are copied from `train_baseline.ipynb` (same seed, same settings), so the test rows are the ones the saved model never saw.

In [ ]:
X = df["HPI"].tolist()
y = df["triage"].to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words="english",
        token_pattern=r"\b[a-zA-Z]{2,}\b",  # ignore ___ and numbers
        min_df=2,
        max_df=0.95,
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",   # critical for rare ESI 4
        random_state=42,
    )),
])

pipeline.fit(X_train, y_train)
test_preds = pipeline.predict(X_test)

# Sanity check against the saved model (not overwritten).
saved = joblib.load(MODEL_PATH)
agreement = np.mean(saved.predict(X_test) == test_preds)
print(f"Test predictions identical to saved baseline_model.pkl: {agreement:.1%}")

## 3. Metrics

In [ ]:
val_preds = pipeline.predict(X_val)

metrics = pd.DataFrame({
    split: {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }
    for split, y_true, y_pred in [("validation", y_val, val_preds), ("test", y_test, test_preds)]
}).T
metrics.round(4)

In [ ]:
print("Classification Report (Test):")
print(classification_report(
    y_test,
    test_preds,
    labels=LABELS,
    target_names=[
        "ESI 1 (Resuscitation)",
        "ESI 2 (Emergent)",
        "ESI 3 (Urgent)",
        "ESI 4 (Less Urgent)",
    ],
    zero_division=0,
))

## 4. Comparison vs simple guesses

How much does the model add over guessing? Both guessers are fitted on the training labels only.
- **Always ESI 3:** predicts the most common level for everyone.
- **Random guess:** picks a level at random, in proportion to how common each level is.

In [ ]:
guessers = {
    "Always ESI 3": DummyClassifier(strategy="most_frequent"),
    "Random guess (class frequencies)": DummyClassifier(strategy="stratified", random_state=42),
}

rows = {}
for name, guesser in guessers.items():
    guesser.fit(X_train, y_train)
    preds = guesser.predict(X_test)
    rows[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "macro_f1": f1_score(y_test, preds, average="macro", labels=LABELS, zero_division=0),
        "weighted_f1": f1_score(y_test, preds, average="weighted", labels=LABELS, zero_division=0),
    }
rows["Baseline model"] = metrics.loc["test"].to_dict()

comparison = pd.DataFrame(rows).T
comparison.round(4)

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_preds, labels=LABELS)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(
    ax=axes[0], cmap="Blues", colorbar=False, values_format="d"
)
axes[0].set_title("Counts")

cm_norm = confusion_matrix(y_test, test_preds, labels=LABELS, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=LABEL_NAMES).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".0%"
)
axes[1].set_title("Share of each actual level (rows sum to 100%)")

for ax in axes:
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
fig.suptitle("Baseline model — test set confusion matrix")
fig.tight_layout()

REPORTS_DIR.mkdir(exist_ok=True)
fig.savefig(REPORTS_DIR / "baseline_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Confidence scores

Confidence is the probability the model gives its chosen level. If confidence is meaningful, predictions made with about 60% confidence should be right about 60% of the time.

In [ ]:
test_proba = pipeline.predict_proba(X_test)
confidence = test_proba.max(axis=1)
correct = test_preds == y_test

bands = pd.cut(confidence, [0, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0])
(
    pd.DataFrame({"confidence": confidence, "correct": correct})
    .groupby(bands, observed=True)
    .agg(
        predictions=("correct", "size"),
        avg_confidence=("confidence", "mean"),
        actual_accuracy=("correct", "mean"),
    )
    .round(3)
)

## 7. Reasoning

`explain(text)` shows which words in the input pushed the model toward its answer. For each word or phrase the model knows, its push toward a level is its TF-IDF value times how much more that level's coefficient is than the average across levels. Words the model has never seen have no effect.

In [ ]:
tfidf = pipeline.named_steps["tfidf"]
clf = pipeline.named_steps["clf"]
terms = tfidf.get_feature_names_out()
# How much each term favours each level over the average level.
relative_coef = clf.coef_ - clf.coef_.mean(axis=0)


def explain(text, top_k=5):
    """Predicted level, probability per level, and the terms that pushed toward it."""
    x = tfidf.transform([text])
    proba = clf.predict_proba(x)[0]
    k = int(proba.argmax())
    contributions = x.toarray()[0] * relative_coef[k]

    order = np.argsort(contributions)[::-1]
    reasons = [(terms[i], float(contributions[i])) for i in order[:top_k] if contributions[i] > 0]

    return {
        "esi": int(clf.classes_[k]),
        "confidence": float(proba[k]),
        "probabilities": {int(c): round(float(p), 3) for c, p in zip(clf.classes_, proba)},
        "reasons": reasons,
        "known_terms": int(x.nnz),
    }


def show(text):
    r = explain(text)
    print(f"Text: {text}")
    print(f"  -> ESI {r['esi']} (confidence {r['confidence']:.0%})")
    print(f"     probabilities: {r['probabilities']}")
    if r["known_terms"] == 0:
        print("     The model recognised none of these words, so the result only reflects its built-in lean toward each level.")
    elif not r["reasons"]:
        print("     No recognised word pushed toward this level more than the others.")
    else:
        print("     pushed toward this level by: " + ", ".join(f"'{t}' (+{w:.2f})" for t, w in r["reasons"]))
    print()

In [ ]:
# Made-up examples only.
samples = [
    "Severe chest pain and difficulty breathing",
    "Lethargy and confusion for two days",
    "Abdominal pain with vomiting",
    "Mild rash on left arm",
    "Found unresponsive at home, hypotensive on arrival",
    "Sudden onset left sided weakness and slurred speech",
    "asdf qwerty",
]
for s in samples:
    show(s)

In [ ]:
# What the model learned overall: terms that most favour each level.
top_terms = pd.DataFrame({
    f"ESI {c}": [terms[i] for i in np.argsort(relative_coef[k])[::-1][:10]]
    for k, c in enumerate(clf.classes_)
})
top_terms.index = range(1, 11)
top_terms

## 8. Summary

In [ ]:
test = metrics.loc["test"]
print("=" * 60)
print("BASELINE MODEL — DEMO 1 SUMMARY")
print("=" * 60)
print(f"Input feature:        HPI text only")
print(f"Model:                TF-IDF + Logistic Regression")
print(f"Train / Val / Test:   {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"Test accuracy:        {test['accuracy']:.4f}")
print(f"Test macro F1:        {test['macro_f1']:.4f}")
print(f"Test weighted F1:     {test['weighted_f1']:.4f}")
print(f"Always ESI 3:         {comparison.loc['Always ESI 3', 'accuracy']:.4f} accuracy")
print(f"Random guess:         {comparison.loc['Random guess (class frequencies)', 'accuracy']:.4f} accuracy")
print(f"Matches saved model:  {agreement:.1%} of test predictions")
print(f"Confusion matrix:     {REPORTS_DIR / 'baseline_confusion_matrix.png'}")
print("=" * 60)